In [ ]:
# Import pandas for working with tables
import pandas as pd

# Upload files from your computer to Google Colab
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import pandas for working with tables
import pandas as pd

# Upload files from your computer to Google Colab
from google.colab import files

uploaded = files.upload()

In [ ]:
# Load the training dataset
train = pd.read_csv("application_train.csv")

# Load the test dataset
test = pd.read_csv("application_test.csv")

# Load the sample submission file
sample_submission = pd.read_csv("sample_submission.csv")

# Display the shapes of the datasets
print("Training data:", train.shape)
print("Test data:", test.shape)
print("Sample submission:", sample_submission.shape)

# Display the first five rows
train.head()

# Count how many examples belong to each TARGET class
print(train["TARGET"].value_counts())

# Show the proportions
print(train["TARGET"].value_counts(normalize=True))

# Display the first few ID values
print(train["SK_ID_CURR"].head())

# Check whether IDs are unique
print("Unique IDs:", train["SK_ID_CURR"].nunique())
print("Total rows:", len(train))

# Display the sample submission
sample_submission.head()

# Display the column names
print(sample_submission.columns)

# Display the shape
print(sample_submission.shape)

Problem 1: Confirmation of competition contents

The purpose of the Home Credit Default Risk competition is to predict whether a client will have difficulty repaying a loan.

The target variable is TARGET.

TARGET = 0 represents a client who did not have payment difficulties according to the competition definition.
TARGET = 1 represents a client who had payment difficulties according to the competition definition.

The training data contains the target variable, while the test data does not.

For the Kaggle submission, we need to create a CSV file containing the applicant identifier SK_ID_CURR and the predicted probability of TARGET = 1.

The competition evaluates predictions using ROC AUC. Therefore, the model should produce a probability or decision score rather than only a hard class label such as 0 or 1.

The class distribution is imbalanced, so accuracy alone would not be an appropriate evaluation measure.

In [ ]:
# Separate the target variable from the input features

X = train.drop(columns=["TARGET"])
y = train["TARGET"]

# Remove the ID column because it is only an identifier
X = X.drop(columns=["SK_ID_CURR"])

# Check the shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Find numerical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns

# Find categorical columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

In [ ]:
from sklearn.model_selection import train_test_split

# Fixed random state so that our experiment can be reproduced
RANDOM_STATE = 42

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Display the shapes
print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)

# Check the target distribution
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Preprocessing for numerical columns
numeric_pipeline = Pipeline([
    # Fill missing numerical values with the median
    ("imputer", SimpleImputer(strategy="median")),

    # Standardize numerical values
    ("scaler", StandardScaler())
])

# Preprocessing for categorical columns
categorical_pipeline = Pipeline([
    # Fill missing categories with the most common category
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert categorical values into numerical columns
    # handle_unknown allows categories appearing only in validation/test
    # to be handled safely
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the two preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the baseline model
model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE
)

# Combine preprocessing and the machine-learning model
baseline_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

# Train the complete baseline pipeline
baseline_pipeline.fit(X_train, y_train)

print("Baseline model training completed.")

In [ ]:
# Predict probabilities for the validation data
y_valid_proba = baseline_pipeline.predict_proba(X_valid)[:, 1]

# Display the first few predicted probabilities
print(y_valid_proba[:10])

from sklearn.metrics import roc_auc_score

# Calculate ROC AUC using the probability of TARGET=1
baseline_auc = roc_auc_score(y_valid, y_valid_proba)

print("Baseline Validation ROC AUC:", baseline_auc)

In [ ]:
#PROBLEM 2
# Display the number of examples in each class
print("Training class counts:")
print(y_train.value_counts())

print("\nValidation class counts:")
print(y_valid.value_counts())

# Display proportions
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))

print("\nValidation class proportions:")
print(y_valid.value_counts(normalize=True))

Problem 2: Baseline model

I separated TARGET from the explanatory variables and removed SK_ID_CURR because it is an identifier.

The training data was divided into training and validation subsets using train_test_split. I used random_state=42 and stratify=y so that the class distribution was preserved between the two subsets.

The preprocessing was performed using a scikit-learn Pipeline and ColumnTransformer.

For numerical features, missing values were replaced with the median and the features were standardized.

For categorical features, missing values were replaced with the most frequent category and categorical values were converted using one-hot encoding.

The baseline model was Logistic Regression with balanced class weights.

The validation ROC AUC was:

ROC AUC = [PUT YOUR ACTUAL SCORE HERE]

The target classes are imbalanced, so ROC AUC was used instead of relying only on accuracy.

In [ ]:
#PROBLEM 3
# Keep the test ID for the submission
test_ids = test["SK_ID_CURR"]

# Remove the ID from the model input
X_test = test.drop(columns=["SK_ID_CURR"])

# Check the shape
print("Test features:", X_test.shape)

# Predict the probability that TARGET = 1
test_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

# Display the first 10 predictions
print(test_proba[:10])

# Create a copy of the sample submission
submission = sample_submission.copy()

# Put our predicted probabilities into the TARGET column
submission["TARGET"] = test_proba

# Display the first few rows
submission.head()

In [ ]:
# Check the submission columns
print("Submission columns:", submission.columns.tolist())

# Check the number of rows
print("Submission rows:", len(submission))

# Compare with sample submission
print("Sample submission rows:", len(sample_submission))

# Check whether IDs are in the same order
print(
    "IDs match:",
    submission["SK_ID_CURR"].equals(sample_submission["SK_ID_CURR"])
)# Save the submission file
submission.to_csv("submission_baseline.csv", index=False)

print("Submission file saved successfully.")

# Read the saved file again to verify it
check_submission = pd.read_csv("submission_baseline.csv")

print(check_submission.head())
print(check_submission.shape)
print(check_submission.columns.tolist())



Problem 3: Estimation on test data

I applied the already-fitted baseline pipeline to application_test.csv.

The model predicted the probability of TARGET = 1 for each test example.

I created the submission using SK_ID_CURR and TARGET, while preserving the original test-row order.

I also checked that the submission has the same column names and number of rows as sample_submission.csv.

The submission was saved without an additional pandas index column as:

submission_baseline.csv

The TARGET column contains predicted probabilities rather than hard class labels.

In [ ]:
#PROBLEM 4
results = []

results.append({
    "Experiment": "Baseline",
    "ROC_AUC": baseline_auc
})

print(results)

# Calculate the percentage of missing values in each column
missing_percentage = train.isnull().mean() * 100

# Sort from highest to lowest
missing_percentage = missing_percentage.sort_values(ascending=False)

# Display the top 20
missing_percentage.head(20)# Find columns with more than 50% missing values
high_missing_columns = missing_percentage[missing_percentage > 50].index

print("Number of columns with more than 50% missing:")
print(len(high_missing_columns))

print("\nColumns:")
print(list(high_missing_columns))
# Remove highly missing columns
X2 = X.drop(columns=high_missing_columns)

# Split using exactly the same random state and stratification
X2_train, X2_valid, y2_train, y2_valid = train_test_split(
    X2,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)




In [ ]:
# Identify numerical and categorical columns for Experiment 2
numeric_features_2 = X2_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features_2 = X2_train.select_dtypes(
    include=["object"]
).columns
# Numerical preprocessing
numeric_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline_2 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor_2 = ColumnTransformer([
    ("num", numeric_pipeline_2, numeric_features_2),
    ("cat", categorical_pipeline_2, categorical_features_2)
])

# Create Experiment 2 model
model_2 = Pipeline([
    ("preprocessing", preprocessor_2),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

# Train Experiment 2
model_2.fit(X2_train, y2_train)

# Predict validation probabilities
y2_proba = model_2.predict_proba(X2_valid)[:, 1]

# Calculate ROC AUC
auc_2 = roc_auc_score(y2_valid, y2_proba)

print("Experiment 2 ROC AUC:", auc_2)


In [ ]:
def add_financial_features(df):
    # Make a copy so the original dataframe is not changed
    df = df.copy()

    # Avoid division by zero by replacing zero with NaN
    income = df["AMT_INCOME_TOTAL"].replace(0, pd.NA)
    credit = df["AMT_CREDIT"].replace(0, pd.NA)
    annuity = df["AMT_ANNUITY"].replace(0, pd.NA)
    goods_price = df["AMT_GOODS_PRICE"].replace(0, pd.NA)

    # Credit amount relative to income
    df["CREDIT_INCOME_RATIO"] = credit / income

    # Annuity relative to income
    df["ANNUITY_INCOME_RATIO"] = annuity / income

    # Goods price relative to income
    df["GOODS_INCOME_RATIO"] = goods_price / income

    # Credit relative to goods price
    df["CREDIT_GOODS_RATIO"] = credit / goods_price

    return df

# Create a new dataset with financial features
X3 = add_financial_features(X)

# Check the new columns
print(X3[[
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_INCOME_RATIO",
    "CREDIT_GOODS_RATIO"
]].head())

X3_train, X3_valid, y3_train, y3_valid = train_test_split(
    X3,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

In [ ]:
def add_age_employment_features(df):
    # Make a copy
    df = df.copy()

    # Convert DAYS_BIRTH into approximate age in years
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

    # Convert DAYS_EMPLOYED into approximate years
    df["EMPLOYED_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25

    # Relationship between employment length and age
    df["EMPLOYMENT_AGE_RATIO"] = (
        df["EMPLOYED_YEARS"] / df["AGE_YEARS"]
    )

    return df

# Add age and employment features
X4 = add_age_employment_features(X)

# Display the engineered features
print(X4[[
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "EMPLOYMENT_AGE_RATIO"
]].head())

In [ ]:
def add_combined_features(df):
    # Start with a copy
    df = df.copy()

    # Financial features

    income = df["AMT_INCOME_TOTAL"].replace(0, pd.NA)
    credit = df["AMT_CREDIT"].replace(0, pd.NA)
    annuity = df["AMT_ANNUITY"].replace(0, pd.NA)
    goods_price = df["AMT_GOODS_PRICE"].replace(0, pd.NA)

    df["CREDIT_INCOME_RATIO"] = credit / income
    df["ANNUITY_INCOME_RATIO"] = annuity / income
    df["GOODS_INCOME_RATIO"] = goods_price / income
    df["CREDIT_GOODS_RATIO"] = credit / goods_price

    # Age and employment features

    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["EMPLOYED_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25

    df["EMPLOYMENT_AGE_RATIO"] = (
        df["EMPLOYED_YEARS"] / df["AGE_YEARS"]
    )

    return df

# Create the combined feature dataset
X5 = add_combined_features(X)

# Display the engineered columns
print(X5[[
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "EMPLOYMENT_AGE_RATIO"
]].head())

In [ ]:
def train_and_evaluate(X_data, y_data, experiment_name):
    """
    Train a Logistic Regression model and return its validation ROC AUC.
    """

    # Split the data using the same settings for every experiment
    X_train_exp, X_valid_exp, y_train_exp, y_valid_exp = train_test_split(
        X_data,
        y_data,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_data
    )

    # Identify numerical and categorical columns
    numeric_cols = X_train_exp.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_cols = X_train_exp.select_dtypes(
        include=["object"]
    ).columns

    # Numerical preprocessing
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine preprocessing
    preprocessing = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

    # Create the complete model pipeline
    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE
        ))
    ])

    # Train the model
    pipeline.fit(X_train_exp, y_train_exp)

    # Predict probability of TARGET=1
    probabilities = pipeline.predict_proba(X_valid_exp)[:, 1]

    # Calculate ROC AUC
    auc = roc_auc_score(y_valid_exp, probabilities)

    print(f"{experiment_name}: ROC AUC = {auc:.4f}")

    return auc, pipeline

In [ ]:
results = []

# Experiment 1: Baseline
auc_1, pipeline_1 = train_and_evaluate(
    X,
    y,
    "Experiment 1 - Baseline"
)

results.append(["Baseline", auc_1])


# Experiment 2: Financial features
X_financial = add_financial_features(X)

auc_2, pipeline_2 = train_and_evaluate(
    X_financial,
    y,
    "Experiment 2 - Financial features"
)

results.append(["Financial features", auc_2])


# Experiment 3: Age and employment features
X_age = add_age_employment_features(X)

auc_3, pipeline_3 = train_and_evaluate(
    X_age,
    y,
    "Experiment 3 - Age/employment features"
)

results.append(["Age/employment features", auc_3])


# Experiment 4: Combined features
X_combined = add_combined_features(X)

auc_4, pipeline_4 = train_and_evaluate(
    X_combined,
    y,
    "Experiment 4 - Combined features"
)

results.append(["Combined features", auc_4])

Privacy and fairness considerations

The Home Credit dataset contains financial and personal attributes. Some variables may be sensitive or may be associated with protected or potentially vulnerable groups.

Examples include demographic information, family-related information, income, employment, and housing-related variables.

A feature having predictive association with TARGET does not mean that the feature causes the outcome. In addition, a higher validation ROC AUC does not by itself justify using a feature in a real credit decision.

For this assignment, the features are used for educational machine-learning experiments. In a real credit decision system, privacy, fairness, legal requirements, explainability, and appropriate feature governance would also need to be considered.